In [0]:
import requests                # conversa com a API (HTTP)

url = "https://brasilapi.com.br/api/feriados/v1/2026"
resposta = requests.get(url, timeout=30)
resposta.raise_for_status()    # se a API cair, para aqui com o erro, em vez de seguir com nada

feriados = resposta.json()     # o JSON vira uma lista do Python: um dicionário por feriado

import json, os
from datetime import datetime

pasta = "/Volumes/lakehouse_vertice/bronze/volume_1/raw/feriados"
os.makedirs(pasta, exist_ok=True)                        # a pasta nova precisa existir

agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")     # ex.: 2026-10-20_19-40-00 (horário UTC)
with open(f"{pasta}/feriados_{agora}.json", "w") as arquivo:
    json.dump(feriados, arquivo, ensure_ascii=False)

sorted(os.listdir(pasta))    



## Passo 3 · Carregar: o raw vira a bronze

Lê **todos** os JSON da pasta: cada execução é uma captura, com o arquivo e a hora em que chegou.

In [ ]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
COMMENT 'Feriados nacionais de 2026, da BrasilAPI. Uma captura por execução do job.'
AS SELECT date AS data, name AS feriado, type AS tipo, weekday AS dia_da_semana,
          _metadata.file_name              AS arquivo,
          _metadata.file_modification_time AS coletado_em
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/raw/feriados/', format => 'json', multiLine => true)
